# Simulate Unitary Cluster Jastrow Circuits

**Abstract**: Simulate depth one unitary cluster Jastrow (UCJ) circuits in polynomial time, notably the $n = 72$ qubit iron sulfur cluster UCJ circuit from [1] https://www.science.org/doi/10.1126/sciadv.adu9991.

## Experiment parameters

In [ ]:
# Parameters of the (L)UCJ ansatz.
half_layer = False                       # If True, appends a final rotation to the circuit as in [1], but makes the energy worse.
alpha_alpha_indices = lambda norb: None  # Use lambda norb: [(p, p + 1) for p in range(norb - 1)] for an LUCJ circuit as in [1]. Use None to run a UCJ circuit with more gates that improves the energy.
alpha_beta_indices  = lambda norb: None  # Use lambda norb: [(p, p) for p in range(0, norb, 4) if p <= 16] for a (truncated) LUCJ circuit as in [1]. Use None to run a UCJ circuit with more gates that improves the energy.

## Setup

### Install packages

In [ ]:
!pip install ffsim tqdm matplotlib --quiet

### Download data

Download the iron sulfur cluster Hamiltonian from the [data repository](https://github.com/jrm874/sqd_data_repository) for [1].

In [ ]:
!curl -O https://raw.githubusercontent.com/jrm874/sqd_data_repository/refs/heads/main/integrals/4Fe-4S/fcidump_Fe4S4_MO.txt

### Imports

In [ ]:
import itertools

import numpy as np
from tqdm.auto import tqdm

import ffsim
import pyscf
import qiskit
import qiskit.visualization
import qiskit.providers.fake_provider


## Problem definition

### Hamiltonian

In [ ]:
fcidump_filename = "fcidump_Fe4S4_MO.txt"  # From the `curl` command above.

# Run Hartree-Fock.
mf_as = pyscf.tools.fcidump.to_scf(fcidump_filename)
mf_as.max_cycle = 100
mf_as.conv_tol = 1e-9
mf_as = mf_as.newton()
mf_as.kernel()
assert mf_as.converged, "SCF did not converge"

# Run CCSD.
ccsd = pyscf.cc.CCSD(mf_as)
eccsd, *_ = ccsd.kernel()

# Extract second-quantized Hamiltonian and Hamiltonian parameters.
constant = pyscf.tools.fcidump.read(fcidump_filename).get("ECORE", 0.0)
h1e = mf_as.get_hcore()
num_orb = h1e.shape[0]
n_qubits = 2 * num_orb
h2e = pyscf.ao2mo.restore(1, mf_as._eri, num_orb)
nelec = pyscf.tools.fcidump.read(fcidump_filename)["NELEC"]

# Display Hamiltonian data.
print(f"Number of spatial orbitals: {num_orb}, Number of qubits: {n_qubits}")
print("CCSD correlation energy:", eccsd)
print("CCSD total energy:", ccsd.e_tot)

Note that CCSD does not converge for this molecule.

### UCJ circuit

In [ ]:
# Build the UCJ Operation.
n_reps_base = 2 if half_layer else 1
base_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=ccsd.t2, n_reps=n_reps_base,  # The polynomial time algorithm applies to one repetition/layer of the UCJ ansatz.
    interaction_pairs=(alpha_alpha_indices(num_orb), alpha_beta_indices(num_orb)),
)
if half_layer:
    ucj_op = ffsim.UCJOpSpinBalanced(
        diag_coulomb_mats=base_op.diag_coulomb_mats[:1],
        orbital_rotations=base_op.orbital_rotations[:1],
        final_orbital_rotation=base_op.orbital_rotations[1].conj().T,
    )
else:
    ucj_op = base_op

In [ ]:
# Build the quantum circuit.
nelec = (nelec // 2, nelec // 2)

qubits = qiskit.QuantumRegister(2 * num_orb, name="q")
circuit = qiskit.QuantumCircuit(qubits)
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)

coupling_map = qiskit.transpiler.CouplingMap.from_full(num_qubits=circuit.num_qubits)
backend = qiskit.providers.fake_provider.GenericBackendV2(
    coupling_map.size(), coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)
compiled = qiskit.transpile(circuit, backend=backend, optimization_level=0)

print(f"UCJ circuit acts on {compiled.num_qubits} qubit(s).")
print(f"Operation count:", compiled.count_ops())

In [ ]:
# Visualize the circuit.
qiskit.visualization.timeline_drawer(
    compiled,
    target=backend.target,
    style=qiskit.visualization.timeline.IQXStandard(**{
        'formatter.general.fig_width': 160,
        'formatter.general.fig_unit_height': 1
    }),
    show_labels=False,
    idle_wires=False,
)

## Simulation

### Step 1: Back-propagate $H$ through final orbital rotations

In [ ]:
def propagate_through_orbital_rotations(h, g, u):
  h_bp = u.conj().T @ h @ u
  g_bp = np.einsum('pi,qj,pqrs,rk,sl->ijkl', u.conj(), u, g, u.conj(), u, optimize=True)
  return h_bp, g_bp

In [ ]:
# Get the final orbital rotations from the UCJ operator.
W = ucj_op.orbital_rotations[0]
Wf = ucj_op.final_orbital_rotation
u = W if Wf is None else Wf @ W

# Back-propagate one-body and two-body tensors through the final orbital rotations.
h_bp, g_bp = propagate_through_orbital_rotations(h1e, h2e, u)

### Step 2: Back-propagate $H$ through Jastrow operators

In [ ]:
def propagate_through_jastrow(same, diff, norb):
    N = 2 * norb

    def phases(nelec_probe):
        da = int(pyscf.fci.cistring.num_strings(norb, nelec_probe[0]))
        db = int(pyscf.fci.cistring.num_strings(norb, nelec_probe[1]))
        v = np.ones(da * db, dtype=complex)
        w = ffsim.apply_diag_coulomb_evolution(
            v,
            (same, diff, same),
            time=-1.0,
            norb=norb,
            nelec=nelec_probe,
        )
        return np.angle(w).reshape(da, db)

    occ1 = [int(o[0]) for o in pyscf.fci.cistring.gen_occslst(range(norb), 1)]
    occ2 = [(int(o[0]), int(o[1])) for o in pyscf.fci.cistring.gen_occslst(range(norb), 2)]

    L = np.zeros(N)
    A = np.zeros((N, N))
    ph_a = phases((1, 0)).ravel()
    ph_b = phases((0, 1)).ravel()
    for i, p in enumerate(occ1):
        L[p] = ph_a[i]
        L[norb + p] = ph_b[i]

    ph_aa = phases((2, 0)).ravel()
    ph_bb = phases((0, 2)).ravel()
    for i, (p, q) in enumerate(occ2):
        A[p, q] = A[q, p] = (ph_aa[i] - L[p] - L[q]) / 2
        A[norb + p, norb + q] = A[norb + q, norb + p] = (ph_bb[i] - L[norb + p] - L[norb + q]) / 2

    ph_ab = phases((1, 1))
    for i, p in enumerate(occ1):
        for j, q in enumerate(occ1):
            A[p, norb + q] = A[norb + q, p] = (ph_ab[i, j] - L[p] - L[norb + q]) / 2

    return A, L

In [ ]:
A_J, L_J = propagate_through_jastrow(ucj_op.diag_coulomb_mats[0][0], ucj_op.diag_coulomb_mats[0][1], num_orb)

### Step 3: Energy calculation via Löwdin formula

In [ ]:
def compute_energy(
    Q: np.typing.NDArray,
    ecore: float,
    h_bp: np.typing.NDArray,
    g_bp: np.typing.NDArray,
    A: np.typing.NDArray,
    L: np.typing.NDArray,
    norb: int,
) -> float:
    """Computes the UCJ1 energy using Löwdin rules for matrix elements of a
    monomial times a Gaussian phase.

    Args:
      Q: (norb x n_occ) Occupied-orbital matrix of e^{-K}|HF>, one spin
          sector (alpha == beta for the spin-balanced ansatz).
      ecore: Energy constant.
      h_bp: (norb x norb) One-body integrals back-propagated through the
          trailing orbital rotation u (i.e. u^dag h u).
      g_bp: (norb,)*4 Two-body integrals back-propagated through u.
      A, L: Scalar and vector phase data from backpropagating the Hamiltonian
          through the Jastrow operation.
      norb: Number of orbitals.
    """
    N = 2 * norb
    Qc = Q.conj()

    # Cache computed terms for speed.
    cache = {}
    cache_cap = max(10_000, int(12e9 // (norb * norb * 16)))   # ~2 GB budget
    def transition(phi):
        key = phi.round(12).tobytes()
        hit = cache.get(key)
        if hit is not None:
            return hit
        d = np.exp(1j * phi)
        S = Qc.T @ (d[:, None] * Q)
        det = np.linalg.det(S)
        rho = (d[:, None] * Q) @ np.linalg.solve(S, Qc.T)
        if len(cache) < cache_cap:
            cache[key] = (det, rho)
        return det, rho

    def dress(Delta):
        phi = -2.0 * (A @ Delta)
        const = np.exp(-1j * (Delta @ A @ Delta + L @ Delta))
        return phi[:norb], phi[norb:], const

    E = ecore + 0j
    # Energy from one-body terms: 2 * sum_pq h[p,q] <a_p^\dagger a_q D>.
    for p in range(norb):
        for q in range(norb):
            Delta = np.zeros(N)
            Delta[p] += 1
            Delta[q] -= 1
            phi_a, phi_b, c = dress(Delta)
            det_a, rho_a = transition(phi_a)
            det_b, _ = transition(phi_b)
            E += 2 * h_bp[p, q] * c * det_a * det_b * rho_a[q, p]

    # Energy from two-body terms.
    n4 = norb ** 4
    bar = tqdm(
        itertools.product(range(norb), repeat=4),
        total=n4,
        desc="compute_energy",
        unit="term",
        mininterval=0.25,
    )
    for (p, q, r, s) in bar:
        g = g_bp[p, q, r, s]

        # Compue same-spin (aa)+(bb) energies.
        Delta = np.zeros(N); Delta[[p, r]] += 1; Delta[[s, q]] -= 1
        phi_a, phi_b, c = dress(Delta)
        (det_a, rho_a), (det_b, _) = transition(phi_a), transition(phi_b)
        wick = rho_a[q, p] * rho_a[s, r] - rho_a[s, p] * rho_a[q, r]
        E += g * c * det_a * det_b * wick

        # Compute opposite-spin (ab)+(ba) energies.
        Delta = np.zeros(N); Delta[p] += 1; Delta[q] -= 1
        Delta[norb + r] += 1; Delta[norb + s] -= 1
        phi_a, phi_b, c = dress(Delta)
        (det_a, rho_a), (det_b, rho_b) = transition(phi_a), transition(phi_b)
        E += g * c * (det_a * rho_a[q, p]) * (det_b * rho_b[s, r])

        # Update progress bar.
        bar.set_postfix(E=f"{E.real:.6f}", refresh=False)

    # Update progress bar with final energy.
    bar.close()

    if abs(E.imag) > 1e-8:
        print(f"warning: Im(E) = {E.imag:.2e}")
    return float(E.real)

In [ ]:
Q = W.conj().T[:, :nelec[0]]  # (num_orbitals x num_occupied_orbitals) matrix.
E_ucj = compute_energy(Q, constant, h_bp, g_bp, A_J, L_J, num_orb)

print(f"UCJ Energy = {E_ucj:.10f} Ha")
print(f"Hartree-Fock Energy = {mf_as.e_tot:.10f} Ha")
print(f"CCSD Energy = {ccsd.e_tot:.10f} Ha")

## References

[1] https://www.science.org/doi/10.1126/sciadv.adu9991.